# UV Index Hybrid Regression — Phase 5
###  Prophet+LGB · CNN-LSTM · Attention-LSTM · Stacking Ensemble

| Model | CPU/GPU | Training scope |
|-------|---------|----------------|
| Prophet+LGB | CPU | 1 model/location, 6-week window |
| CNN-LSTM | **GPU** | Pooled all locations, full history |
| Attention-LSTM | **GPU** | Pooled all locations, full history |
| Stacking Ensemble | CPU/GPU | Pooled all locations, full history |


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from prophet import Prophet
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

import torch
import torch.nn as nn
import joblib

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Configuration ──────────────────────────────────────────────────────────────
TARGET       = 'uv_index'
TRAIN_WEEKS  = 6
FORECAST_DAYS = 14
SEQ_LEN      = 48
BATCH        = 256
DEMO_LOC     = 'hcm'
MODEL_DIR    = Path('../../models')
MODEL_DIR.mkdir(exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────


Device: cuda


## 1. Load Data

In [2]:
import sys
sys.path.insert(0, str(Path('../../').resolve()))
from src.config import CORE_14_FEATURES

DATA_DIR = Path('../../data/processed/features')
MODEL_DIR = Path('../../models') / 'legacy'
MODEL_DIR.mkdir(exist_ok=True, parents=True)

df = pd.read_csv(DATA_DIR / 'features_regression.csv', parse_dates=['timestamp'])
if 'uv_source' in df.columns:
    real_mask = df['uv_source'].isin(['weatherbit', 'open_meteo']) & df['uv_index'].notna()
    df = df[real_mask].copy()
    print(f'Filtered to real UV data: {len(df)} rows')

df = df.sort_values(['location_id', 'timestamp']).reset_index(drop=True)

LOCATIONS = sorted(df['location_id'].unique())
print(f'Locations ({len(LOCATIONS)}): {LOCATIONS}')
print(f'Shape: {df.shape}')

# ── Feature columns: strictly CORE_14_FEATURES ──────────────────────────────
FEATURE_COLS = [c for c in CORE_14_FEATURES if c in df.columns]
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

# ── 70/15/15 temporal split boundaries ──────────────────────────────────────
_min = df['timestamp'].min()
_max = df['timestamp'].max()
_total_days = (_max - _min).days
TRAIN_END = _min + pd.Timedelta(days=int(_total_days * 0.70))
VAL_END   = TRAIN_END + pd.Timedelta(days=int(_total_days * 0.15))
print(f'Split: train→{TRAIN_END.date()} | val→{VAL_END.date()} | test→{_max.date()}')

# ── Pool-level train/val/test DataFrames ─────────────────────────────────────
train_df = df[df['timestamp'] < TRAIN_END].copy()
val_df   = df[(df['timestamp'] >= TRAIN_END) & (df['timestamp'] < VAL_END)].copy()
test_df  = df[df['timestamp'] >= VAL_END].copy()

fill_vals = train_df[FEATURE_COLS].median()
scaler    = StandardScaler()
X_tr_raw  = train_df[FEATURE_COLS].fillna(fill_vals).values
X_val_raw = val_df[FEATURE_COLS].fillna(fill_vals).values
X_te_raw  = test_df[FEATURE_COLS].fillna(fill_vals).values

X_tr_s  = scaler.fit_transform(X_tr_raw)
X_val_s = scaler.transform(X_val_raw)
X_te_s  = scaler.transform(X_te_raw)

y_tr_all  = train_df[TARGET].values
y_val_all = val_df[TARGET].values
y_te_all  = test_df[TARGET].values


Filtered to real UV data: 386992 rows
Locations (8): ['cangio', 'cuchi', 'dalat', 'hcm', 'longhai', 'nhabe', 'thuduc', 'vungtau']
Shape: (386992, 38)
Feature columns (10): ['cos_solar_zenith', 'doy_sin', 'temperature_2m', 'relative_humidity_2m', 'cloud_cover', 'solar_cloud_interaction', 'ozone_anomaly', 'pressure_msl', 'wind_speed_10m', 'altitude_m']
Split: train→2023-01-20 | val→2024-09-04 | test→2026-04-21


## 2. Evaluation Helper

In [3]:
results = {}

def evaluate(name, y_true, y_pred, split='val'):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 0.1 ))) * 100

    results[f'{name}_{split}'] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE': mape}
    print(f'  {name} [{split}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.3f}')
    return mae, rmse, r2, mape


---
## Hybrid 2 — Prophet + LightGBM Residuals
> Prophet captures trend + daily/yearly seasonality. LightGBM learns residuals from weather features.
> **Per-location model. 6-week training window.**

In [4]:
EXOG_PROPHET = FEATURE_COLS   # ✅ All 14 core features (was 3)
print(f'Training Prophet+LGB for all {len(LOCATIONS)} locations (14 regressors)...')

prophet_models, lgb_residual_models = {}, {}
prophet_preds_val = {}

for loc in LOCATIONS:
    sub_all = df[df['location_id'] == loc].copy().sort_values('timestamp')
    sub_tr  = train_df[train_df['location_id'] == loc].copy().sort_values('timestamp')
    sub_v   = val_df[val_df['location_id']     == loc].copy().sort_values('timestamp')
    sub_te   = test_df[test_df['location_id']     == loc].copy().sort_values('timestamp')

    avail_exog = [c for c in EXOG_PROPHET if c in sub_tr.columns]
    fill_loc   = sub_tr[avail_exog].median()

    sub_tr_clean = sub_tr[['timestamp', TARGET] + avail_exog].fillna(fill_loc)
    sub_v_clean  = sub_v[['timestamp', TARGET]  + avail_exog].fillna(fill_loc)
    sub_te_clean  = sub_te[['timestamp', TARGET]  + avail_exog].fillna(fill_loc)

    # Scale regressors (Prophet handles its own y-scaling internally)
    loc_scaler = StandardScaler()
    sub_tr_clean[avail_exog] = loc_scaler.fit_transform(sub_tr_clean[avail_exog].values)
    sub_v_clean[avail_exog]  = loc_scaler.transform(sub_v_clean[avail_exog].values)
    sub_te_clean[avail_exog]  = loc_scaler.transform(sub_te_clean[avail_exog].values)

    # Prophet format
    train_pr = sub_tr_clean.rename(columns={'timestamp': 'ds', TARGET: 'y'})
    val_pr   = sub_v_clean.rename(columns={'timestamp': 'ds', TARGET: 'y'})
    test_pr   = sub_te_clean.rename(columns={'timestamp': 'ds', TARGET: 'y'})

    # Fit Prophet with all 14 regressors
    m = Prophet(growth='linear', daily_seasonality=True,
                yearly_seasonality=True, weekly_seasonality=False)
    for col in avail_exog:
        m.add_regressor(col)
    m.fit(train_pr)

    # In-sample residuals
    in_pred = m.predict(train_pr[['ds'] + avail_exog])['yhat'].values
    resids  = train_pr['y'].values - in_pred

    # LGB on residuals (14 features)
    lgb_r = lgb.LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.05,
                               random_state=SEED, verbose=-1)
    lgb_r.fit(train_pr[avail_exog].values, resids)
    lgb_residual_models[loc] = lgb_r

    # Forecast val
    prophet_train = in_pred
    lgb_train = lgb_r.predict(train_pr[avail_exog].values)
    pred_train = (prophet_train + lgb_train).clip(min=0)

    prophet_val = m.predict(val_pr[['ds'] + avail_exog])['yhat'].values
    lgb_val     = lgb_r.predict(val_pr[avail_exog].values)
    pred_val    = (prophet_val + lgb_val).clip(min=0)

    prophet_test = m.predict(test_pr[['ds'] + avail_exog])['yhat'].values
    lgb_test = lgb_r.predict(test_pr[avail_exog].values)
    pred_test = (prophet_test + lgb_test).clip(min=0)

    prophet_preds_val[loc] = (val_pr['y'].values, val_pr['ds'].values, pred_val)
    print(f'  {loc.upper()} fitted')
    evaluate(f'Prophet+LGB_{loc}', train_pr['y'].values, pred_train, split='train')
    evaluate(f'Prophet+LGB_{loc}', val_pr['y'].values, pred_val, split='val')
    evaluate(f'Prophet+LGB_{loc}', test_pr['y'].values, pred_test, split='test')

    # Save and free memory at the end of the location loop!
    import joblib
    joblib.dump(m, MODEL_DIR / f'hybrid_prophet_{loc}.joblib', compress=('lzma', 3))
    prophet_models[loc] = None
    del m
    import gc; gc.collect()

mae_list  = [results[f'Prophet+LGB_{l}_train']['MAE']  for l in LOCATIONS]
rmse_list = [results[f'Prophet+LGB_{l}_val']['RMSE'] for l in LOCATIONS]
r2_list   = [results[f'Prophet+LGB_{l}_val']['R2']   for l in LOCATIONS]
print(f'\n→ Avg val  MAE={np.mean(mae_list):.4f}  RMSE={np.mean(rmse_list):.4f}  R2={np.mean(r2_list):.3f}')


Training Prophet+LGB for all 8 locations (14 regressors)...


12:13:30 - cmdstanpy - INFO - Chain [1] start processing


12:13:38 - cmdstanpy - INFO - Chain [1] done processing


  CANGIO fitted
  Prophet+LGB_cangio [train]  MAE=0.3081  RMSE=0.5851  R2=0.900
  Prophet+LGB_cangio [val]  MAE=0.4299  RMSE=0.7389  R2=0.864
  Prophet+LGB_cangio [test]  MAE=0.4198  RMSE=0.6937  R2=0.850


12:13:45 - cmdstanpy - INFO - Chain [1] start processing


...

  THUDUC fitted
  Prophet+LGB_thuduc [train]  MAE=0.2975  RMSE=0.5537  R2=0.907
  Prophet+LGB_thuduc [val]  MAE=0.4349  RMSE=0.7134  R2=0.872
  Prophet+LGB_thuduc [test]  MAE=0.4644  RMSE=0.6757  R2=0.853


12:15:21 - cmdstanpy - INFO - Chain [1] start processing


12:15:31 - cmdstanpy - INFO - Chain [1] done processing


  VUNGTAU fitted
  Prophet+LGB_vungtau [train]  MAE=0.3047  RMSE=0.5594  R2=0.906
  Prophet+LGB_vungtau [val]  MAE=0.4875  RMSE=0.7715  R2=0.851
  Prophet+LGB_vungtau [test]  MAE=0.5521  RMSE=0.7518  R2=0.821



→ Avg val  MAE=0.2929  RMSE=0.7291  R2=0.863


---
## Hybrid 3 & 4 — CNN-LSTM · Attention-LSTM
> PyTorch sequence models trained on **all locations pooled together**. Uses full training history (≤2024). GPU-accelerated if available.

In [5]:
# ── DL/Stacking pooled data uses train_df/val_df/test_df already defined in Cell 3 ──
print('Preparing pooled sequence data for DL / Stacking models...')
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
N_FEAT = len(FEATURE_COLS)
print(f'Features: {N_FEAT}')

import torch
from torch.utils.data import TensorDataset, DataLoader

# Build sequences per location to avoid boundary artefacts
SEQ_LEN = 48
BATCH   = 64
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def build_sequences(split_df, X_scaled, y_arr, seq_len):
    X_seqs, y_seqs = [], []
    for loc in LOCATIONS:
        mask = split_df['location_id'].values == loc
        lX = X_scaled[mask]
        ly = y_arr[mask]
        for i in range(seq_len - 1, len(lX)):
            X_seqs.append(lX[i-seq_len+1:i+1])
            y_seqs.append(ly[i])
    return np.array(X_seqs), np.array(y_seqs)

seq_X_tr, seq_y_tr = build_sequences(train_df, X_tr_s, y_tr_all, SEQ_LEN)
seq_X_v,  seq_y_v  = build_sequences(val_df,   X_val_s, y_val_all, SEQ_LEN)
seq_X_te, seq_y_te = build_sequences(test_df,  X_te_s, y_te_all, SEQ_LEN)

def make_dl(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, drop_last=shuffle)

train_dl = make_dl(seq_X_tr, seq_y_tr, shuffle=True)
val_dl   = make_dl(seq_X_v,  seq_y_v)
test_dl  = make_dl(seq_X_te, seq_y_te)

print(f'Sequences — train: {seq_X_tr.shape}  val: {seq_X_v.shape}  test: {seq_X_te.shape}')
print(f'Device: {DEVICE}')


Preparing pooled sequence data for DL / Stacking models...
Train: 270,881 | Val: 58,677 | Test: 57,434
Features: 10


Sequences — train: (270505, 48, 10)  val: (58301, 48, 10)  test: (57058, 48, 10)
Device: cuda


In [6]:
class CNNLSTM(nn.Module):
    def __init__(self, input_dim, conv_out=64, hidden_dim=256):
        super().__init__()
        self.conv = nn.Conv1d(input_dim, conv_out, kernel_size=3, padding=1)
        self.bn   = nn.BatchNorm1d(conv_out)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(conv_out, hidden_dim, num_layers=2,
                            batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x):
        x = self.relu(self.bn(self.conv(x.transpose(1, 2)))).transpose(1, 2)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)

class AttentionLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.lstm   = nn.LSTM(input_dim, hidden_dim, num_layers=2,
                              batch_first=True, dropout=0.2)
        self.attn_w = nn.Linear(hidden_dim, 1)
        self.head   = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(),
                                    nn.Dropout(0.2), nn.Linear(64, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        attn   = torch.softmax(self.attn_w(out), dim=1)
        ctx    = torch.sum(attn * out, dim=1)
        return self.head(ctx).squeeze(-1)


In [7]:
def train_torch(model, name, train_dl, val_dl,
                epochs=40, patience=8, lr=1e-3):
    model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    crit  = nn.MSELoss()
    best_val, p_ctr = float('inf'), 0
    save_path = str(MODEL_DIR / f'{name}_hybrid.pt')

    for epoch in range(1, epochs+1):
        model.train()
        for Xb, yb in train_dl:
            pred = model(Xb.to(DEVICE))
            loss = crit(pred, yb.to(DEVICE))
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for Xb, yb in val_dl:
                val_losses.append(crit(model(Xb.to(DEVICE)), yb.to(DEVICE)).item())
        avg_val = np.mean(val_losses)
        sched.step(avg_val)

        if avg_val < best_val:
            best_val, p_ctr = avg_val, 0
            torch.save(model.state_dict(), save_path)
        else:
            p_ctr += 1
        if epoch % 10 == 0 or epoch == 1:
            print(f'  {name} epoch {epoch}/{epochs}  val_loss={avg_val:.4f}  patience={p_ctr}/{patience}')
        if p_ctr >= patience:
            print(f'  Early stopping at epoch {epoch}'); break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    return model

def predict_torch(model, dl):
    preds = []
    with torch.no_grad():
        for Xb, _ in dl:
            preds.append(model(Xb.to(DEVICE)).cpu().numpy())
    return np.concatenate(preds)


In [8]:
print('Training CNN-LSTM...')
cnn_lstm = train_torch(CNNLSTM(N_FEAT), 'cnn_lstm', train_dl, val_dl)

train_eval_dl = make_dl(seq_X_tr, seq_y_tr, shuffle=False)

pred_cnn_train = predict_torch(cnn_lstm, train_eval_dl)
pred_cnn_val  = predict_torch(cnn_lstm, val_dl)
pred_cnn_test = predict_torch(cnn_lstm, test_dl)
evaluate('CNN-LSTM',  seq_y_tr,  pred_cnn_train, split='train')
evaluate('CNN-LSTM',  seq_y_v,  pred_cnn_val, split='val')
evaluate('CNN-LSTM',  seq_y_te,  pred_cnn_test, split='test')



Training CNN-LSTM...


  cnn_lstm epoch 1/40  val_loss=0.4126  patience=0/8


  cnn_lstm epoch 10/40  val_loss=0.4839  patience=6/8


  Early stopping at epoch 12


  CNN-LSTM [train]  MAE=0.2020  RMSE=0.4896  R2=0.926
  CNN-LSTM [val]  MAE=0.2763  RMSE=0.6416  R2=0.894
  CNN-LSTM [test]  MAE=0.2610  RMSE=0.6047  R2=0.881


(0.2610234964120157,
 np.float64(0.6047052868904829),
 0.8806217535528004,
 np.float64(29.242851386094216))

In [9]:
print('Training Attention-LSTM...')
attn_lstm = train_torch(AttentionLSTM(N_FEAT), 'attn_lstm', train_dl, val_dl)

train_eval_dl = make_dl(seq_X_tr, seq_y_tr, shuffle=False)


pred_attn_train = predict_torch(attn_lstm, train_eval_dl)
pred_attn_val  = predict_torch(attn_lstm, val_dl)
pred_attn_test = predict_torch(attn_lstm, test_dl)
evaluate('Attention-LSTM',  seq_y_tr,  pred_attn_train, split='train')
evaluate('Attention-LSTM',  seq_y_v,  pred_attn_val, split='val')
evaluate('Attention-LSTM',  seq_y_te,  pred_attn_test, split='test')


Training Attention-LSTM...


  attn_lstm epoch 1/40  val_loss=0.5718  patience=0/8


  attn_lstm epoch 10/40  val_loss=0.4874  patience=7/8


  Early stopping at epoch 11


  Attention-LSTM [train]  MAE=0.2164  RMSE=0.5048  R2=0.921
  Attention-LSTM [val]  MAE=0.2823  RMSE=0.6461  R2=0.893
  Attention-LSTM [test]  MAE=0.2718  RMSE=0.6147  R2=0.877


(0.27181973950951427,
 np.float64(0.614673028685417),
 0.8766537421540489,
 np.float64(32.847704829409324))

---
## Hybrid 5 — Stacking Ensemble
> Level-0: RF, XGB, LGB, CatBoost trained on pooled all-location features. > Level-1: Ridge meta-learner on OOF predictions.

In [10]:
print('Training Stacking Ensemble (OOF, 5-fold)...')
N_FOLDS = 5
kf = TimeSeriesSplit(n_splits=N_FOLDS)

l0_models = {
    'rf':  lambda: RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    'xgb': lambda: xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                     random_state=SEED, verbosity=0, n_jobs=-1),
    'lgb': lambda: lgb.LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                                      random_state=SEED, verbose=-1, n_jobs=-1),
    'cat': lambda: CatBoostRegressor(iterations=200, depth=5, learning_rate=0.05,
                                      random_seed=SEED, verbose=0),
}

oof_preds   = np.zeros((len(X_tr_s), len(l0_models)))
best_l0     = {name: None for name in l0_models}
best_scores = {name: float('inf') for name in l0_models}

for fold_i, (tr_idx, oof_idx) in enumerate(kf.split(X_tr_s)):
    X_fold_tr, X_fold_oof = X_tr_s[tr_idx], X_tr_s[oof_idx]
    y_fold_tr, y_fold_oof = y_tr_all[tr_idx], y_tr_all[oof_idx]
    for j, (name, factory) in enumerate(l0_models.items()):
        m = factory()
        m.fit(X_fold_tr, y_fold_tr)
        pred_oof = m.predict(X_fold_oof)
        oof_preds[oof_idx, j] = pred_oof
        fold_rmse = np.sqrt(mean_squared_error(y_fold_oof, pred_oof))
        if fold_rmse < best_scores[name]:
            best_scores[name] = fold_rmse
            best_l0[name] = m

print('Training Ridge meta-learner...')
meta = Ridge(alpha=1.0).fit(oof_preds, y_tr_all)

print('retraining base models oin full training set')
full_l0 = {}
for name, factory in l0_models.items():
    m = factory()
    m.fit(X_tr_s, y_tr_all)
    full_l0[name] = m
    print(f" {name} trained on full set")

train_preds_l0  = np.column_stack([full_l0[n].predict(X_tr_s) for n in l0_models])
val_preds_l0    = np.column_stack([full_l0[n].predict(X_val_s) for n in l0_models])
test_preds_l0   = np.column_stack([full_l0[n].predict(X_te_s) for n in l0_models])

pred_stack_train = meta.predict(train_preds_l0)
pred_stack_val = meta.predict(val_preds_l0)
pred_stack_test = meta.predict(test_preds_l0)

evaluate('Stacking', y_tr_all, pred_stack_train, split='train')
evaluate('Stacking', y_val_all, pred_stack_val, split='val')
evaluate('Stacking', y_te_all,  pred_stack_test, split='test')

del full_l0, train_preds_l0, val_preds_l0, test_preds_l0
import gc; gc.collect()


Training Stacking Ensemble (OOF, 5-fold)...


Training Ridge meta-learner...
retraining base models oin full training set


 rf trained on full set


 xgb trained on full set


 lgb trained on full set


 cat trained on full set


  Stacking [train]  MAE=0.2619  RMSE=0.3106  R2=0.970
  Stacking [val]  MAE=0.5552  RMSE=0.8350  R2=0.820
  Stacking [test]  MAE=0.5360  RMSE=0.7761  R2=0.803


190

## 6. Model Comparison

In [11]:
# Aggregate per-location stats for Hybrids 1 & 2
def avg_results(prefix, split='val'):
    keys = [k for k in results if k.startswith(f'{prefix}_') and k.endswith(f'_{split}')]
    if not keys: return {}
    return {m: np.mean([results[k][m] for k in keys]) for m in ['MAE','RMSE','R2']}

summary_rows = []
for name, split in [
    ('Prophet+LGB',   'val'),  ('Prophet+LGB',   'test'),
    ('CNN-LSTM',      'val'),  ('CNN-LSTM',      'test'),
    ('Attention-LSTM','val'),  ('Attention-LSTM','test'),
    ('Stacking',      'val'),  ('Stacking',      'test'),
]:
    if f'{name}_{split}' in results:
        row = {'Model': name, 'Split': split, **results[f'{name}_{split}']}
    else:
        avg = avg_results(name, split)
        if avg:
            row = {'Model': name, 'Split': split, **avg}
        else:
            continue
    summary_rows.append(row)

summ_df = pd.DataFrame(summary_rows).set_index(['Model', 'Split'])
display(summ_df.round(4).sort_values(['Split', 'RMSE']))


,,MAE,RMSE,R2,MAPE
Model,Split,,,,
CNN-LSTM,test,0.2610,0.6047,0.8806,29.2429
Attention-LSTM,test,0.2718,0.6147,0.8767,32.8477
Prophet+LGB,test,0.4758,0.6964,0.8402,NaN
Stacking,test,0.5360,0.7761,0.8025,255.9748
CNN-LSTM,val,0.2763,0.6416,0.8943,20.5976
Attention-LSTM,val,0.2823,0.6461,0.8928,21.4508
Prophet+LGB,val,0.4456,0.7291,0.8627,NaN
Stacking,val,0.5552,0.8350,0.8203,240.2062


## 7. Save Models

In [12]:
## export resuult to CSV
import pandas as pd
from pathlib import Path
results_list = []
for key, metrics in results.items():
    if '_' in key:
        parts = key.rsplit('_', 1)
        if len(parts) == 2:
            model_name, split = parts
        else:
            model_name = key
            split = 'Unknown'
    else:
        model_name = key
        split = 'Unknown'

    results_list.append({
        'notebook': 'regression',
        'model': model_name,
        'split': split,
        'MAE': metrics.get('MAE'),
        'RMSE': metrics.get('RMSE'),
        'R2': metrics.get('R2'),
        'MAPE': metrics.get('MAPE')
    })

results_df = pd.DataFrame(results_list)

results_dir = Path('../../results')
results_dir.mkdir(exist_ok=True)
results_df.to_csv(results_dir / 'hybrid_results.csv', index=False)
print('SAVED REGRESSION RESULTS')

SAVED REGRESSION RESULTS


In [13]:
import pickle
from pathlib import Path

checkpoint_dir = Path('../../results/checkpoints')
checkpoint_dir.mkdir(exist_ok=True, parents=True)
with open(checkpoint_dir / 'hybrid_results_dict.pkl', 'wb') as f:
    pickle.dump(results, f)

print(f"results dictionary saved to {checkpoint_dir / 'hybrid_results_dict.pkl'}")

results dictionary saved to ../../results/checkpoints/hybrid_results_dict.pkl


In [14]:
for loc in LOCATIONS:
        joblib.dump(lgb_residual_models[loc], MODEL_DIR / f'hybrid_lgb_resid_{loc}.joblib', compress=('lzma', 3))

for name, m in best_l0.items():
    joblib.dump(m, MODEL_DIR / f'hybrid_l0_{name}.joblib', compress=('lzma', 3))
joblib.dump(meta, MODEL_DIR / 'hybrid_meta_ridge.joblib', compress=('lzma', 3))

print(f'Saved all hybrid models to {MODEL_DIR.resolve()}')

Saved all hybrid models to /home/dat-vu/Desktop/UV_analysis/models/legacy
